In [12]:
import torch
import torch.nn as nn
from torch.nn import Sequential, Identity
import torch.nn.functional as fnn
import numpy as np

In [5]:
from resnet.models.VGG import VGG19, device

In [7]:
B,H,W,C = 1,224,224,3
im = torch.tensor(np.ones((B,C,H,W)),dtype=torch.float32).to(device)

## VGG-19 model

In [ ]:
model = VGG19(3).to(device)

In [ ]:
fnn.softmax(model(im), dim=-1)

## ResNet

In [ ]:
class Additive(Sequential):
    def forward(self, input):
        return sum(L(input) for L in self)
    
def ResBlock(C, stride=1):
    ksize, pad = 3,1
    Cin, Cout = C,stride*C
    if stride==1:
        skip_connection = Identity()
    else:
        skip_connection = Sequential(
            nn.Conv2d(Cin,Cout, kernel_size=1, bias=False, stride=stride),
            nn.BatchNorm2d(Cout),
        )
    return Sequential(
        Additive(
            skip_connection,
            Sequential(
                nn.Conv2d(Cin,Cout,ksize, padding=pad, bias=False, stride=stride),
                nn.BatchNorm2d(Cout),
                nn.ReLU(),
                nn.Conv2d(Cout,Cout,ksize, padding=pad, bias=False, stride=1),
                nn.BatchNorm2d(Cout),
            ),
        ),
        nn.ReLU()
    )

ResNet18 = Sequential(
    # conv1
    nn.Conv2d(3,64, kernel_size=7, stride=2, padding=3, bias=False),
    nn.BatchNorm2d(64),
    nn.ReLU(),
    # conv2_x
    nn.MaxPool2d(kernel_size=3, stride=2),
    ResBlock(64),
    ResBlock(64),
    # conv3_x
    ResBlock(64, stride=2),
    ResBlock(128),
    # conv4_x
    ResBlock(128, stride=2),
    ResBlock(256),
    # conv5_x
    ResBlock(256, stride=2),
    ResBlock(512),
    # global avg_pool
    nn.AdaptiveAvgPool2d((1,1)),
    nn.Flatten(),
    # fc+softmax
    nn.Linear(512,1000),
    nn.Softmax()
)

In [59]:
ResNet18.to(device)(im)

/Users/ovantzos/Developer/mleng/resnet/.venv/lib/python3.12/site-packages/torch/nn/modules/module.py:1783: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


tensor([[0.0009, 0.0015, 0.0018, 0.0005, 0.0004, 0.0013, 0.0024, 0.0017, 0.0005,
         0.0011, 0.0009, 0.0011, 0.0005, 0.0011, 0.0009, 0.0011, 0.0005, 0.0007,
         0.0004, 0.0015, 0.0014, 0.0010, 0.0011, 0.0009, 0.0008, 0.0005, 0.0006,
         0.0009, 0.0012, 0.0013, 0.0004, 0.0005, 0.0008, 0.0009, 0.0014, 0.0015,
         0.0008, 0.0019, 0.0011, 0.0016, 0.0008, 0.0018, 0.0007, 0.0006, 0.0009,
         0.0006, 0.0013, 0.0008, 0.0008, 0.0005, 0.0005, 0.0011, 0.0009, 0.0011,
         0.0023, 0.0010, 0.0019, 0.0010, 0.0005, 0.0007, 0.0005, 0.0006, 0.0015,
         0.0015, 0.0004, 0.0008, 0.0014, 0.0014, 0.0004, 0.0005, 0.0009, 0.0013,
         0.0014, 0.0003, 0.0010, 0.0019, 0.0014, 0.0011, 0.0005, 0.0013, 0.0006,
         0.0023, 0.0017, 0.0014, 0.0004, 0.0011, 0.0006, 0.0007, 0.0012, 0.0008,
         0.0016, 0.0013, 0.0013, 0.0013, 0.0008, 0.0007, 0.0008, 0.0013, 0.0007,
         0.0012, 0.0007, 0.0010, 0.0005, 0.0009, 0.0009, 0.0006, 0.0009, 0.0012,
         0.0019, 0.0008, 0.0